# Chapter 0 — Setup

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/00_setup.ipynb)

This notebook installs [AGAMA](https://github.com/GalacticDynamics-Oxford/Agama) — the
library we use for most of the tutorial — and checks that it reproduces analytic results for a
Plummer sphere. Along the way it introduces three things you will use in every later chapter:
AGAMA's **units**, its **`Potential`** object, and its **orbit integrator**.

## 0.1 Installing AGAMA

AGAMA ("Action-based Galaxy Modelling Architecture", Vasiliev 2019) is written in C++ with a
Python interface. It has to be compiled, which takes several minutes. On Google Colab we skip
that: the cell below downloads a ready-made, pre-compiled package ("wheel") built for Colab by
this tutorial's repository, which takes about 10 seconds.

In [ ]:
import sys, importlib.util

# importlib.util.find_spec('agama') returns None if the package is not installed,
# so the install runs only once per Colab session.
if importlib.util.find_spec('agama') is None:
    # The wheel must match the Python version: 'cp313' for Python 3.13, 'cp312' for 3.12.
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    # Lines starting with '!' are shell commands (a Jupyter/Colab feature, not Python);
    # {v} inserts the Python variable v into the command.
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"

import agama
print(agama.__version__)

Running locally rather than on Colab? Install AGAMA once following its
[INSTALL notes](https://github.com/GalacticDynamics-Oxford/Agama/blob/master/INSTALL); the cell
above then just imports it.

## 0.2 Units: tell AGAMA once, at the start

AGAMA works internally in units where the gravitational constant $G = 1$. You choose how those
internal units map onto physical ones by calling `agama.setUnits` with **three** independent
scales (mass, length, velocity — time then follows). The usual galactic choice is

| quantity | unit |
|---|---|
| mass | $1\,M_\odot$ |
| length | 1 kpc |
| velocity | 1 km/s |
| time (derived) | 1 kpc / (1 km/s) $= 0.978$ Gyr |

In these units $G = 4.30 \times 10^{-6}\ \mathrm{kpc\,(km/s)^2}/M_\odot$; AGAMA stores it as `agama.G`.

Three rules to remember:
1. **Call `setUnits` once, before creating any AGAMA object.** Potentials and distribution
   functions remember the unit system they were created in; changing units afterwards silently
   mis-scales them (AGAMA prints a warning).
2. **Numbers carry no units.** AGAMA takes and returns plain `numpy` arrays. A position `[8, 0, 0]`
   means 8 kpc only because we said so here; it is up to you to stay consistent.
3. **Times are in the derived unit** (0.978 Gyr here), not in Gyr or Myr. To integrate for 1 Gyr,
   pass `time = 1000 / agama.getUnits()['time']` (`getUnits` reports the time unit in Myr).

If you never call `setUnits`, AGAMA uses $G = 1$ with no physical meaning attached ("N-body
units") — convenient for theory, as in Chapters 1–2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

agama.setUnits(mass=1, length=1, velocity=1)   # 1 Msun, 1 kpc, 1 km/s
print('G in these units          :', agama.G)
print('time unit in Myr          :', agama.getUnits()['time'])

## 0.3 A first potential: the Plummer sphere

`agama.Potential` builds a gravitational potential from keyword arguments. `type` names one of
AGAMA's built-in models (`'Plummer'`, `'NFW'`, `'MiyamotoNagai'`, `'Logarithmic'`, ... — see §2.2
of the AGAMA reference manual); the other arguments set its parameters, in the units chosen above.

The Plummer sphere of mass $M$ and scale radius $b$ has

$$ \Phi(r) = -\frac{GM}{\sqrt{r^2 + b^2}} , $$

where $r$ is the distance from the centre and $\Phi$ the gravitational potential (energy per unit mass).
We take $M = 10^{10}\,M_\odot$ (a small galaxy bulge) and $b = 1$ kpc.

In [ ]:
M, b = 1e10, 1.0                                    # Msun, kpc
pot = agama.Potential(type='Plummer', mass=M, scaleRadius=b)

# AGAMA evaluates everything at Cartesian points (x, y, z): one point is a list of 3 numbers,
# many points are an array of shape (N, 3).
print('Phi at the centre  :', pot.potential([0, 0, 0]), '(km/s)^2')
print('expected -G M / b  :', -agama.G * M / b)

## 0.4 Circular speed: AGAMA vs the formula

A star on a circular orbit of radius $R$ needs a centripetal acceleration $v_c^2/R$ equal to the
inward pull of gravity, $|F_R| = \partial\Phi/\partial R$:

$$ v_c(R) = \sqrt{R\,\frac{\partial\Phi}{\partial R}} = \sqrt{-R\,F_R} , $$

where $R$ is the distance from the centre in the plane of the orbit and $F_R = -\partial\Phi/\partial R$ the radial
component of the gravitational force per unit mass (negative: it points inwards).

For the Plummer sphere this gives $v_c^2 = GMR^2/(R^2+b^2)^{3/2}$.

**AGAMA has no dedicated circular-speed function.** As AGAMA's own tutorial
(`py/tutorial_potential_orbits.ipynb` in the AGAMA repository) does, we build it from
`pot.force`, which returns the acceleration $\mathbf F = -\nabla\Phi$ at each point.
Related built-ins you will meet later: `pot.enclosedMass(r)` (mass inside radius $r$),
`pot.Rcirc(L=...)` or `pot.Rcirc(E=...)` (radius of the circular orbit with given angular momentum
or energy) and `pot.Tcirc(E)` (its period).

In [ ]:
r = np.logspace(-2, 2, 200)                         # 200 radii from 0.01 to 100 kpc, evenly spaced in log r

# Points on the x-axis: np.column_stack glues three length-200 arrays into a (200, 3) array.
points = np.column_stack([r, 0 * r, 0 * r])
F = pot.force(points)                               # shape (200, 3): (Fx, Fy, Fz) at each point
vc_agama = np.sqrt(-r * F[:, 0])                    # on the x-axis, F_R = F_x  (column 0)

vc_exact = np.sqrt(agama.G * M * r**2 / (r**2 + b**2)**1.5)

# For a spherical system there is a second route, v_c^2 = G M(<r) / r, via AGAMA's enclosedMass:
vc_mass = np.sqrt(agama.G * pot.enclosedMass(r) / r)

plt.loglog(r, vc_exact, 'k-', lw=3, alpha=0.3, label='formula')
plt.loglog(r, vc_agama, 'r--', label=r'AGAMA: $\sqrt{-R F_R}$')
plt.loglog(r, vc_mass, 'b:', label=r'AGAMA: $\sqrt{G M(<r)/r}$')
plt.axvline(np.sqrt(2) * b, color='gray', ls=':', label=r'peak at $\sqrt{2}\,b$')
plt.xlabel('r [kpc]'); plt.ylabel(r'$v_c$ [km/s]'); plt.legend(); plt.show()

err = np.max(np.abs(vc_agama / vc_exact - 1))
err_mass = np.max(np.abs(vc_mass / vc_exact - 1))
print(f'max relative error in v_c: from force {err:.1e}, from enclosed mass {err_mass:.1e}')
assert err < 1e-10 and err_mass < 1e-10

**Figure 0.1.** Circular speed $v_c(r)$ of the Plummer sphere (log–log axes; $r$ in kpc, $v_c$ in km/s): the formula (thick grey), AGAMA via $\sqrt{-R F_R}$ (red dashed) and via $\sqrt{GM(<r)/r}$ (blue dotted). The vertical line marks the peak at $\sqrt2\,b$.

Both routes agree with the formula to machine precision (Figure 0.1). They are equivalent only for a
spherical system; for a flattened galaxy (a disk, Chapter 4) $GM(<r)/r$ is no longer the circular
speed, and the force route is the one to use.

## 0.5 Integrating an orbit

`agama.orbit` integrates the equations of motion $\dot{\mathbf x} = \mathbf v$,
$\dot{\mathbf v} = -\nabla\Phi$ with a high-order adaptive integrator (Chapter 2 explains how such
integrators work and how to test them).

- `ic` — initial conditions, 6 numbers $(x, y, z, v_x, v_y, v_z)$ in kpc and km/s;
- `time` — total integration time, in the derived time unit (0.978 Gyr);
- `trajsize` — number of equally spaced moments at which to store the trajectory;
- `dtype` — storage precision. AGAMA stores trajectories as 32-bit floats by default (about
  7 significant digits) to save memory; pass `dtype=float` (64-bit) when you need precise values,
  e.g. to check energy conservation.

It returns two arrays: the times (length `trajsize`) and the phase-space positions
(shape `(trajsize, 6)`).

Two quantities must stay constant along an orbit in a static spherical potential, which makes
them good tests of the integration:
- the **energy** per unit mass $E = \tfrac12 v^2 + \Phi(\mathbf x)$, with $v = |\mathbf v|$ the speed and $\mathbf x$ the position;
- the **radial action** $J_r$ — a measure of how far the star oscillates in radius (Chapter 9).
  `agama.ActionFinder(pot)` builds an object that, called on positions/velocities, returns the
  actions $(J_r, J_z, J_\phi)$ as an `(N, 3)` array.

In [ ]:
# start at x = 1 kpc moving mostly in y (150 km/s) with a small z component (30 km/s);
# integrate for 10 time units (about 10 Gyr), storing 2001 snapshots in double precision
times, traj = agama.orbit(potential=pot, ic=[1, 0, 0, 0, 150, 30], time=10.0, trajsize=2001, dtype=float, verbose=False)

# traj[:, :3] are positions, traj[:, 3:] velocities; np.sum(..., axis=1) sums over x, y, z for each snapshot
E = pot.potential(traj[:, :3]) + 0.5 * np.sum(traj[:, 3:]**2, axis=1)

Jr = agama.ActionFinder(pot)(traj[::200])[:, 0]     # every 200th snapshot; column 0 is J_r

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(traj[:, 0], traj[:, 1], lw=0.5)
ax[0].set_aspect('equal'); ax[0].set_xlabel('x [kpc]'); ax[0].set_ylabel('y [kpc]')
ax[0].set_title('orbit seen from above')
ax[1].plot(times * agama.getUnits()['time'] / 1000, E / E[0] - 1)  # convert time to Gyr for the plot
ax[1].set_xlabel('time [Gyr]'); ax[1].set_ylabel(r'$E/E_0 - 1$'); ax[1].set_title('energy error')
plt.tight_layout(); plt.show()

print(f'max |dE/E| = {np.max(np.abs(E/E[0]-1)):.1e},  std(Jr)/mean(Jr) = {np.std(Jr)/np.mean(Jr):.1e}')
assert np.max(np.abs(E/E[0]-1)) < 1e-5 and np.std(Jr)/np.mean(Jr) < 1e-4
print('AGAMA works.')

**Figure 0.2.** Left: an orbit in the Plummer potential integrated with `agama.orbit` for 10 time units, seen from above ($x$–$y$ plane). Right: relative energy error $E/E_0 - 1$ against time in Gyr.

Seen from above (Figure 0.2, left), the orbit fills a ring between its pericentre and apocentre (Chapters 1–2 explain
why). Energy is conserved to better than one part in a million over 10 Gyr (right), but notice that the
error **grows steadily** rather than oscillating. AGAMA's default integrator (`method='dop853'`, an
8th-order Runge–Kutta scheme) is very accurate per step but not *symplectic*; Chapter 2 shows what
that means and why leapfrog behaves differently. For most purposes this drift is negligible; if you
need tighter conservation, lower the `accuracy` argument of `agama.orbit` (default `1e-8`): for this
orbit, `accuracy=1e-10` gives $|\Delta E/E| \approx 4\times10^{-9}$ and `1e-12` gives $4\times10^{-11}$.

You are ready for Chapter 1.

## References

- E. Vasiliev (2019), "AGAMA: action-based galaxy modelling architecture", MNRAS 482, 1525:
  https://doi.org/10.1093/mnras/sty2672
- AGAMA reference manual: https://github.com/GalacticDynamics-Oxford/Agama/blob/master/doc/reference.pdf